# ORCA-X Kaggle GPU Runner

This notebook is the project-owned Kaggle GPU entry point. It clones the exact Git branch, rebuilds the generated historical dataset when needed, verifies CUDA, and runs one ORCA-X ML job.

**Kaggle settings:** enable **GPU** and **Internet** before running.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/Sayan260106/HackHeritage.git'
REPO_REF = os.environ.get('ORCA_X_REF', 'refinement-36-coastal-domain-adaptation')
REPO_DIR = Path('/kaggle/working/HackHeritage')
SCRIPT = os.environ.get('ORCA_X_SCRIPT', 'ml/src/refinement36_coastal_domain_adaptation.py')

print('Repository:', REPO_URL)
print('Branch:', REPO_REF)
print('Script:', SCRIPT)

In [ ]:
# Always operate from /kaggle/working before deleting the repository copy.
%cd /kaggle/working
!rm -rf HackHeritage
!git clone --branch "{REPO_REF}" --single-branch "{REPO_URL}" HackHeritage
%cd /kaggle/working/HackHeritage
!git rev-parse --abbrev-ref HEAD
!git rev-parse HEAD
!test -f "{SCRIPT}" && echo 'OK   target script found'

In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip install -q -r ml/requirements-colab.txt

import torch
import xgboost as xgb

print('PyTorch CUDA available:', torch.cuda.is_available())
print('XGBoost version:', xgb.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
import os
os.environ['ORCA_X_DEVICE'] = 'cuda'
os.environ['ORCA_X_N_JOBS'] = '2'

DATA = '/kaggle/working/HackHeritage/ml/data/processed/orca_historical_marine_risk.parquet'
if not os.path.exists(DATA):
    print('Canonical processed dataset missing; rebuilding from real Open-Meteo historical sources...')
    !python ml/src/colab_prepare.py
else:
    print('Canonical processed dataset already exists:', DATA)

assert os.path.exists(DATA), DATA
print('Dataset ready:', DATA)

In [ ]:
import os
os.environ['ORCA_X_DEVICE'] = 'cuda'
os.environ['ORCA_X_N_JOBS'] = '2'

print('Running:', SCRIPT)
!python ml/src/colab_gpu_runner.py "{SCRIPT}"

In [ ]:
#!python ml/src/colab_gpu_runner.py ml/src/refinement36_coastal_domain_adaptation.py

In [ ]:
# %cd /kaggle/working/HackHeritage

# import pandas as pd

# base = "ml/models/refinement36"

# for f in [
#     "per_coast_strategy_summary.csv",
#     "spatial_2025_domain_adaptation_results.csv",
#     "adaptation_budgets.csv",
# ]:
#     print("\n" + "="*100)
#     print(f)
#     print("="*100)
#     df = pd.read_csv(f"{base}/{f}")
#     print(df.to_string(index=False))

In [17]:
import os

os.chdir("/kaggle/working")
print("Current directory:", os.getcwd())
print("HackHeritage exists:", os.path.exists("/kaggle/working/HackHeritage"))

Current directory: /kaggle/working
HackHeritage exists: False


In [18]:
import os
import shutil
import subprocess

os.chdir("/kaggle/working")

repo = "/kaggle/working/HackHeritage"

# Remove any incomplete clone
if os.path.exists(repo):
    shutil.rmtree(repo)

print("Cloning HackHeritage...")
result = subprocess.run(
    [
        "git", "clone",
        "--depth", "1",
        "--single-branch",
        "--branch", "main",
        "https://github.com/Sayan260106/HackHeritage.git",
        repo
    ],
    text=True
)

print("Git exit code:", result.returncode)

Cloning HackHeritage...


Cloning into '/kaggle/working/HackHeritage'...


Git exit code: 128


fatal: could not read Username for 'https://github.com': No such device or address
fatal: expected flush after ref listing
